# From Crystal Structure to Conformation: Automated OpenMM Simulation & Discovery Pipeline

[![Open in JupyterLab](https://img.shields.io/badge/Launch-JupyterLab-orange.svg)](https://jupyter.org)
[![Python 3.10+](https://img.shields.io/badge/python-3.10+-blue.svg)](https://www.python.org/)

One of MolScope's most distinctive capabilities is its **native 3-stage simulation-to-discovery continuum**. Researchers can supply a raw crystal structure or AlphaFold PDB and MolScope will:
1. Automatically fix, protonate at pH 7.4, and parameterize the system in OpenMM.
2. Generate transparent, reviewable execution plans (`openmm_plan.json`) with dry-run byte and frame estimates.
3. Integrate Langevin molecular dynamics with checkpointing and provenance tracking.
4. Chain directly into multi-family featurization, DPA state discovery, and ML feature prioritization.


## 1. Imports and Verification


In [ ]:
import json
from pathlib import Path
import tempfile
import matplotlib.pyplot as plt
import pandas as pd

from molscope.core.project import (
    ProjectSpec, SystemSpec, EnsembleSpec, ReplicaSpec, OpenMMSimulationSpec
)
from molscope.simulation.openmm import (
    write_openmm_plan, inspect_openmm_plan, execute_openmm_plan
)

print("OpenMM simulation planning modules imported successfully.")


## 2. Defining an OpenMM Simulation Protocol

We specify a complete simulation protocol declaratively:
- **Force field**: Amber14-SB with GBN2 implicit solvent (`implicit/gbn2.xml`).
- **Thermostat**: Langevin integrator at 300 K with 1.0 ps⁻¹ friction.
- **Timestep**: 2.0 fs with hydrogen bonds constrained.
- **Phases**: Energy minimization (L-BFGS), NVT thermal equilibration, and Langevin production dynamics.


In [ ]:
protocol = OpenMMSimulationSpec(
    timestep_femtoseconds=2.0,
    temperature_kelvin=300.0,
    friction_per_picosecond=1.0,
    nonbonded_method="NoCutoff",
    constraints="HBonds",
    minimization_max_iterations=100,
    equilibration_steps=500,
    production_steps=5000,          # 10 ps production for demonstration
    report_interval_steps=100,       # Save every 0.2 ps (50 frames total)
    platform="CPU",                  # Use 'CUDA' or 'OpenCL' on GPU instances
    forcefield_files=("amber14-all.xml", "implicit/gbn2.xml"),
    ph=7.4,
    seed=42,
)

print("Simulation Protocol configured:")
print(f"  Temperature: {protocol.temperature_kelvin} K")
print(f"  Production:  {protocol.production_steps} steps ({protocol.production_duration_ns:.3f} ns)")
print(f"  Save Rate:   Every {protocol.report_interval_steps} steps ({protocol.save_interval_ps:.2f} ps)")


## 3. Creating a Project Specification and Generating the Execution Plan

We create a `ProjectSpec` referencing our input structure and generate a portable, reviewable plan.


In [ ]:
# Locate test protein fixture
fixture_pdb = Path("tests/fixtures/1A22.pdb")
if not fixture_pdb.is_file():
    fixture_pdb = Path("../tests/fixtures/1A22.pdb")

run_dir = Path("results/simulation_demo")
run_dir.mkdir(parents=True, exist_ok=True)

system = SystemSpec(
    system_id="growth_factor_hgh",
    topology=fixture_pdb,
    simulation_protocol=protocol,
    ensembles=(
        EnsembleSpec(
            ensemble_id="nvt_production",
            replicas=(
                ReplicaSpec(replica_id="rep_01", trajectory=run_dir / "sim_output/trajectory.dcd"),
            ),
        ),
    ),
)

project = ProjectSpec(project_id="demo_sim_to_discovery", systems=(system,))

plan_path, launcher_path = write_openmm_plan(project, run_dir / "sim_output")
print(f"Plan generated:     {plan_path}")
print(f"Launcher script:    {launcher_path}")


## 4. Dry-Run Plan Inspection & Byte/Frame Estimation

Before spending expensive GPU/CPU compute, MolScope inspects the plan without importing heavyweight simulation dependencies:
- Verifies input file existence and SHA-256 digests.
- Calculates exact expected frame counts.
- Estimates trajectory file size in bytes.


In [ ]:
with open(plan_path) as f:
    plan = json.load(f)

preview = inspect_openmm_plan(plan)
job = preview["jobs"][0]

print(f"Job ID:             {job['job_id']}")
print(f"Status:             {job['status']}")
print(f"Input Structure:    {job['input_structure']}")
print(f"Expected Frames:    {job['estimate']['expected_frames']}")
print(f"Estimated Atom Count: {job['estimate']['input_atom_count']}")
print(f"Approx DCD Bytes:   {job['estimate']['approximate_dcd_bytes']:,} bytes")


## 5. Checkpointed Execution & Segment Resumption

MolScope guarantees reproducible simulations:
- Generates an immutable `execution_provenance.json` containing SHA-256 hashes of the topology, force field, serialized system XML, and protocol parameters.
- Writes binary checkpoints (`checkpoint.chk`) periodically.
- Interrupted or crashed runs can be resumed into immutable child segments without altering parent replica history.


In [ ]:
print("To execute this simulation plan on your machine:")
print(f"  python -m molscope.simulation.openmm {plan_path}")
print("\nOr use the single-command CLI:")
print(f"  molscope run-pdb --pdb {fixture_pdb} --out-dir results/my_pdb_run --steps 10000 --states 2")


## 6. Chaining Directly into MolScope Discovery

Once the simulation completes, the output trajectory (`trajectory.dcd`) and prepared topology (`prepared.pdb`) are loaded into a `Study` with zero manual intervention:

```python
study = Study(project_id="sim_discovery", out_dir="results/discovery_results")
study.add_system(
    system_id="hgh_system",
    topology=run_dir / "sim_output/prepared_topology.pdb",
    trajectories=[run_dir / "sim_output/trajectory.dcd"],
)
study.configure_imaging(unwrap=True, center_selection="protein")
study.configure_features(["protein.ca_distances_contacts", "protein.secondary_structure_dssp"])
study.run()
```
